# Step 8 — Cropped cache (v2)

**Settings: Accelerator = `None` (CPU), Internet = `Off`.** Attach the competition data.

## Why rebuild rather than crop the existing cache

Cropping `cache.npy` and upscaling would gain nothing — the detail was already discarded when we resized the full field of view down to 160x160. To actually gain resolution the crop has to happen **before** the resize, on the original DICOM pixels.

Same output size, same 8 GB, same 160x160. But now those 160 pixels cover the joint instead of the joint plus a wide margin of black background and soft tissue.

## Why this should help the labels we're weakest on

A meniscus is a few millimetres thick. Across a full ~160 mm field of view rendered at 160 pixels, that's roughly 2-3 pixels — right at the limit of what a convolutional network can resolve. Crop away the background and the same structure spans 5-6 pixels.

Your weakest gold-holdout labels are exactly the thin structures. This targets them directly.

## How the crop is chosen

A **tissue bounding box**, computed per series, not per slice:

1. Take the middle third of slices (the most anatomically representative).
2. Threshold at the 60th percentile of non-zero intensity to separate tissue from background and noise.
3. Take the bounding box of that tissue mask across those slices.
4. Expand by 8% margin, force it square (so aspect ratio is preserved rather than squashed), clamp to image bounds.
5. Apply that single box to every slice in the series.

One box per series, never per slice — a box that jitters slice to slice would make the volume incoherent and destroy the sequence model's input.

**Fallbacks matter.** If the mask is empty or degenerate, we fall back to the central 80% crop, and failing that to the full frame. A crop that silently returns garbage on 5% of series would be worse than no crop at all.

## Discipline reminder

Whatever this file does, `rsna-knee-06-inference` must do identically. Train/test preprocessing mismatch throws no error and silently destroys your score.

### 1 · Setup

In [ ]:
import os, glob, time, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT  = "/kaggle/working"

PX, NSL, NSER = 160, 24, 3
CROP_MARGIN   = 0.08     # expand the tissue box by this fraction
CROP_PCTL     = 60       # intensity percentile separating tissue from background

train = pd.read_csv(f"{ROOT}/train.csv")
ser   = pd.read_csv(f"{ROOT}/train_series.csv")
studies = train["StudyInstanceUID"].tolist()

print(f"studies {len(studies):,} | series {len(ser):,}")
print(f"cache ({len(studies)}, {NSER}, {NSL}, {PX}, {PX}) uint8 = "
      f"{len(studies)*NSER*NSL*PX*PX/1e9:.1f} GB")
print("cores:", os.cpu_count())

### 2 · Series selection — unchanged from v1

In [ ]:
def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in ser.groupby("StudyInstanceUID")}
print("selection done")

### 3 · The crop

Returns a square box in original-pixel coordinates. Everything about it is deterministic — no randomness, so training and inference produce identical crops for identical input.

In [ ]:
def tissue_box(vol):
    """vol: (Z, H, W) float32 in original resolution. Returns (y0, y1, x0, x1)."""
    H, W = vol.shape[1:]
    z0, z1 = len(vol) // 3, max(len(vol) * 2 // 3, len(vol) // 3 + 1)
    mid = vol[z0:z1]

    nz = mid[mid > 0]
    if nz.size < 100:
        return centre_box(H, W)
    thr  = np.percentile(nz, CROP_PCTL)
    mask = (mid > thr).any(0)

    ys, xs = np.where(mask)
    if len(ys) < 50:
        return centre_box(H, W)

    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    h, w = y1 - y0, x1 - x0
    if h < H * 0.15 or w < W * 0.15:          # degenerate box -> fall back
        return centre_box(H, W)

    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    half   = max(h, w) * (0.5 + CROP_MARGIN)   # square, with margin
    half   = min(half, min(H, W) / 2)

    y0 = int(max(0, cy - half)); y1 = int(min(H, cy + half))
    x0 = int(max(0, cx - half)); x1 = int(min(W, cx + half))
    if y1 - y0 < 16 or x1 - x0 < 16:
        return centre_box(H, W)
    return y0, y1, x0, x1


def centre_box(H, W):
    s = int(min(H, W) * 0.8) // 2
    cy, cx = H // 2, W // 2
    return max(0, cy - s), min(H, cy + s), max(0, cx - s), min(W, cx + s)

print("crop ready")

### 4 · Decoder

Identical to v1 except for one change: slices are decoded at **original resolution**, the tissue box is computed once across the series, and only then does the resize to 160x160 happen. That ordering is the whole point.

In [ ]:
def load_series(dirpath, laterality_hint=None):
    files = os.listdir(dirpath)
    if not files:
        return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas:
        return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]
        r, c = iop[:3], iop[3:]
        n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos)
    metas = [metas[i] for i in order]
    take  = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    # ---- decode at ORIGINAL resolution (v1 resized here; v2 does not) ----
    raw, shape = [], None
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
            if shape is None: shape = a.shape
            if a.shape != shape:
                a = cv2.resize(a, (shape[1], shape[0]), interpolation=cv2.INTER_AREA)
        except Exception:
            a = np.zeros(shape or (PX, PX), np.float32)
        raw.append(a)
    vol = np.stack(raw)

    # ---- crop, THEN resize ----
    y0, y1, x0, x1 = tissue_box(vol)
    vol = vol[:, y0:y1, x0:x1]
    vol = np.stack([cv2.resize(s, (PX, PX), interpolation=cv2.INTER_AREA) for s in vol])

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0:
            vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    lat = laterality_hint or next((m[4] for m in metas if m[4]), "")
    if str(lat).upper().startswith("L"):
        lr_axis = int(np.argmax([abs(v[0]) for v in axes]))
        vol = np.flip(vol, axis=lr_axis)

    return (vol * 255).astype(np.uint8)

def study_paths(suid):
    return [(f"{ROOT}/train_series/{suid}/{c}" if c else None) for c in chosen.get(suid, [None]*3)]

print("decoder ready")

### 5 · Visual check BEFORE committing an hour

This is the cell that matters. Look at the two rows: v1 full-frame on top, v2 cropped below. The knee should fill noticeably more of the frame in the bottom row, and the joint should still be fully inside it.

If the crop is cutting off anatomy, or if some series come back nearly black, stop here — do not commit.

In [ ]:
import matplotlib.pyplot as plt

V1 = glob.glob("/kaggle/input/notebooks/*/*/cache.npy")
old = np.load(V1[0], mmap_mode="r") if V1 else None

test_ids = [0, 7, 21, 40]
fig, ax = plt.subplots(2, len(test_ids)*2, figsize=(19, 5.5))
for n, si in enumerate(test_ids):
    p = study_paths(studies[si])[0]
    new = load_series(p) if p and os.path.isdir(p) else None
    for j, sl in enumerate([8, 14]):
        col = n*2 + j
        if old is not None:
            ax[0, col].imshow(old[si, 0, sl], cmap="gray", vmin=0, vmax=255)
        ax[0, col].axis("off")
        if new is not None:
            ax[1, col].imshow(new[sl], cmap="gray", vmin=0, vmax=255)
        ax[1, col].axis("off")
ax[0, 0].set_title("v1  full frame", loc="left", fontsize=11)
ax[1, 0].set_title("v2  cropped", loc="left", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# quantitative check: how much frame does the crop actually remove?
ratios = []
for si in range(0, 400, 8):
    p = study_paths(studies[si])[0]
    if not (p and os.path.isdir(p)): continue
    try:
        fs = sorted(os.listdir(p))[:1]
        a = pydicom.dcmread(os.path.join(p, fs[0])).pixel_array.astype(np.float32)
        v = np.stack([a])
        y0, y1, x0, x1 = tissue_box(np.stack([a]*3))
        ratios.append(((y1-y0) * (x1-x0)) / (a.shape[0] * a.shape[1]))
    except Exception:
        pass
r = np.array(ratios)
print(f"sampled {len(r)} series")
print(f"cropped area as fraction of original: mean {r.mean():.2f}  p10 {np.percentile(r,10):.2f}  p90 {np.percentile(r,90):.2f}")
print(f"=> effective linear resolution gain: {1/np.sqrt(r.mean()):.2f}x")
print("\n(if mean is ~0.95 the crop is doing nothing; if <0.3 it may be cutting anatomy)")

### 6 · Build

In [ ]:
CACHE = f"{OUT}/cache_crop.npy"
DONE  = f"{OUT}/done_crop.npy"
SHAPE = (len(studies), NSER, NSL, PX, PX)

if not os.path.exists(CACHE):
    np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=SHAPE)
    np.save(DONE, np.zeros(len(studies), np.uint8))
    print("created", CACHE)

done = np.load(DONE)
print(f"already done: {done.sum():,}/{len(studies):,}")

def work(i):
    arr = np.lib.format.open_memmap(CACHE, mode="r+")
    got = 0
    for k, p in enumerate(study_paths(studies[i])):
        if p and os.path.isdir(p):
            try:
                v = load_series(p)
                if v is not None:
                    arr[i, k] = v; got += 1
            except Exception:
                pass
    del arr
    return i, got

todo = [i for i in range(len(studies)) if not done[i]]
print(f"to process: {len(todo):,}\n")

t0 = time.time()
with Pool(4) as pool:
    for n, (i, got) in enumerate(pool.imap_unordered(work, todo, chunksize=8), 1):
        done[i] = 1
        if n % 200 == 0 or n == len(todo):
            np.save(DONE, done)
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el/n:.2f}s/study  eta {(len(todo)-n)*el/n/60:.0f} min", flush=True)
np.save(DONE, done)
print(f"\ncomplete: {done.sum():,}/{len(studies):,} in {(time.time()-t0)/60:.0f} min")

### 7 · Verify

In [ ]:
arr = np.load(CACHE, mmap_mode="r")
rng = np.random.default_rng(0)
chk = rng.choice(len(studies), 300, replace=False)

nz = np.array([[arr[i, k].max() > 0 for k in range(NSER)] for i in chk])
print(f"series slots populated : {nz.mean():.1%}")
print(f"  sag {nz[:,0].mean():.1%} | cor {nz[:,1].mean():.1%} | ax {nz[:,2].mean():.1%}")
print(f"studies with >=1 series: {nz.any(1).mean():.1%}")
print(f"intensity mean {arr[chk[:50]].mean():.1f}  std {arr[chk[:50]].std():.1f}")

pd.DataFrame({"StudyInstanceUID": studies, "done": done}).to_csv(f"{OUT}/cache_crop_index.csv", index=False)

i = int(np.where(done > 0)[0][0])
fig, ax = plt.subplots(3, 8, figsize=(18, 7))
for k in range(3):
    for j in range(8):
        ax[k, j].imshow(arr[i, k, j*3], cmap="gray", vmin=0, vmax=255)
        ax[k, j].axis("off")
    ax[k, 0].set_title(["sagittal","coronal","axial"][k], loc="left", fontsize=11)
plt.suptitle("cropped cache - one study, every 3rd slice")
plt.tight_layout(); plt.show()